<a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/LLM_Explanation_Lab/LLM_Explanation_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Can we trust an AI's explanation?
**Dr. Ali Louati**

In this lab you will:
1. Build a small **traffic junction** with three stages (groups of lanes that get green together).
2. Let a pretend **AI advisor** pick the next green stage and **give a reason**.
3. **Check** if the reason is true, using the data.
4. Add a **safety shield**: simple rules that stop very bad decisions.

How to use it: click **Runtime → Run all**. Then read each step and do the ✏️ tasks.

> Note: this is a teaching toy. The traffic is random (made up). It copies the kind of mistakes we saw in a real study of a Riyadh junction, but it is **not** that study's code.

## Step 1. The junction
Every 5 seconds we know two things for each stage:
- **queue**: how many cars are waiting
- **wait**: how many seconds since this stage last had green

In [ ]:
import random
import re
import matplotlib.pyplot as plt

random.seed(1)
STAGES = ["A: north-south", "B: east-west", "C: left turns"]

def random_state():
    """Make one random picture of the junction."""
    return {s: {"queue": random.randint(0, 25), "wait": random.choice(range(0, 181, 5))} for s in STAGES}

state = random_state()
for s, v in state.items():
    print(f"{s:16}  queue = {v['queue']:2} cars   wait = {v['wait']:3} s")

## Step 2. A pretend AI advisor
The advisor picks a stage and writes a reason, like a chatbot would.

Like the language model in our study, it **sounds sure**, but it often uses the number of the stage it *picked*, not the real longest wait.

In [ ]:
def ai_advisor(state):
    """A pretend AI. It likes the stage with the most cars,
    but it always says it picked the 'longest wait'."""
    pick = max(state, key=lambda s: state[s]["queue"])
    reason = f"Give green to {pick}: it has the longest wait, {state[pick]['wait']} s."
    return pick, reason

pick, reason = ai_advisor(state)
print("AI says:", reason)

## Step 3. Check the reason
A reason is **faithful** only if it matches the data.
The AI said "longest wait, N s". So we check: is N really the longest wait?

In [ ]:
def check_reason(state, reason):
    """Return True if a 'longest wait, N s' claim is true, False if it is false,
    and None if there is no such claim to check."""
    m = re.search(r"longest wait[^0-9]*(\d+)", reason)
    if not m:
        return None
    claimed = int(m.group(1))
    real = max(v["wait"] for v in state.values())
    return claimed == real

print("AI says:     ", reason)
print("Real longest:", max(v["wait"] for v in state.values()), "s")
print("Is the reason true?", check_reason(state, reason))

### Do it 200 times
One example is not enough. Let us check many decisions.

In [ ]:
results = []
for _ in range(200):
    st = random_state()
    _, r = ai_advisor(st)
    results.append(check_reason(st, r))

false_share = results.count(False) / len(results)
print(f"False reasons: {false_share:.0%} of {len(results)} decisions")

plt.bar(["True reason", "False reason"], [results.count(True), results.count(False)], color=["#43A047", "#DC2626"])
plt.title("Does the AI's reason match the data?")
plt.ylabel("Number of decisions")
plt.show()

✏️ **Task 1.** Change `ai_advisor` so it picks the stage with the **longest wait**. Run the cells again.
How many reasons are false now? Why?

## Step 4. A safety shield
We cannot trust the reasons, so we add **rules** that the AI cannot break:
- no stage may wait more than **MAX_RED** seconds (here 150 s).

If a stage waits too long, the shield gives it green, whatever the AI says.

In [ ]:
MAX_RED = 150   # seconds

def shield(pick, state):
    """Keep the AI's pick unless a stage has waited too long."""
    worst = max(state, key=lambda s: state[s]["wait"])
    if state[worst]["wait"] >= MAX_RED:
        return worst, True          # the shield steps in
    return pick, False

def simulate(use_shield, steps=360, seed=7):
    """Run the junction for 30 minutes (360 steps of 5 s).
    Cars arrive at random; the green stage lets 3 cars go per step."""
    rnd = random.Random(seed)
    st = {s: {"queue": 0, "wait": 0} for s in STAGES}
    arrive = {"A: north-south": 1.8, "B: east-west": 1.6, "C: left turns": 0.2}  # cars per step
    worst_wait, overrides = [], 0
    for _ in range(steps):
        for s in STAGES:                      # new cars arrive
            st[s]["queue"] += sum(rnd.random() < arrive[s] / 2 for _ in range(2))
        pick, _ = ai_advisor(st)
        if use_shield:
            pick, used = shield(pick, st)
            overrides += used
        for s in STAGES:                      # green stage moves, others wait
            if s == pick:
                st[s]["queue"] = max(0, st[s]["queue"] - 3)
                st[s]["wait"] = 0
            else:
                st[s]["wait"] += 5
        worst_wait.append(max(v["wait"] for v in st.values()))
    return worst_wait, overrides

no_shield, _ = simulate(use_shield=False)
with_shield, n = simulate(use_shield=True)
print(f"Longest red without shield: {max(no_shield)} s")
print(f"Longest red with shield:    {max(with_shield)} s   (shield stepped in {n} times)")

minutes = [i * 5 / 60 for i in range(len(no_shield))]
plt.plot(minutes, no_shield, label="AI alone", color="#DC2626")
plt.plot(minutes, with_shield, label="AI + safety shield", color="#43A047")
plt.axhline(MAX_RED, ls="--", color="grey", label="MAX_RED limit")
plt.xlabel("Time (minutes)"); plt.ylabel("Longest red time now (s)")
plt.title("The shield stops stages from waiting forever")
plt.legend(); plt.show()

✏️ **Task 2.** Change `MAX_RED` to 90 and then to 240. What happens to the longest red time and to the number of times the shield steps in?

## Step 5. Your turn with a real chatbot
1. Copy the junction table printed below.
2. Paste it into any chatbot (ChatGPT, Gemini, Copilot…) and ask:
   *"Which stage should get green next, and why? Mention the longest wait in seconds."*
3. Paste the chatbot's answer into `chatbot_answer` and run the cell.

In [ ]:
random.seed(2026)
my_state = random_state()
print("Stage            | Queue (cars) | Wait (s)")
for s, v in my_state.items():
    print(f"{s:16} | {v['queue']:12} | {v['wait']}")

In [ ]:
chatbot_answer = "Paste the chatbot's answer here. Example: Give green to C: it has the longest wait, 40 s."

verdict = check_reason(my_state, chatbot_answer)
if verdict is None:
    print("No 'longest wait' number found. Read the answer yourself and check each number by hand.")
else:
    print("The chatbot's claim is", "TRUE ✅" if verdict else "FALSE ❌")

## Quick check
1. An AI's reason sounds clear and confident. Does that mean it is true?
2. What is the difference between a **plausible** reason and a **faithful** reason?
3. Why do we need a safety shield even if the AI is usually good?

<details><summary>🔑 Answers (try first)</summary>

1. No. Sounding sure is not the same as being correct. Always check the numbers.
2. Plausible = it *sounds* right. Faithful = it *matches the data* and the real cause of the decision.
3. Because a few bad decisions can be dangerous. Simple rules give a guarantee the AI cannot give.
</details>

**Takeaway:** treat an AI's explanation as a claim to test, not a reason to trust.

*Related research: Albanyan, Louati & Louati (2026), "LLM and Deep Learning in the Loop of Disturbed Traffic Control", Algorithms 19(7), 550.*